In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm

import glob
from processing import *
from classical_estimates import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/phi-fdt-deadpix_20250915T140003_V202609091415C_0569150100.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [4]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

['/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240330T050009_V202608262158C_0463300100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240926T114503_V202608262134C_0469260100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241016T113003_V202608262111C_0470160100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241027T233003_V202608262048C_0470270100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241202T123003_V202608262027C_0472020100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250119T210009_V202608262003C_0561190100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250310T080009_V202608261939C_0563100100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250923T000503_V202608261853C_0569230100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20260310T040003_V202608261828C_0663100

In [5]:
i = -3
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [6]:
folder_blos = '/home/ulyanov/data/pmi/20250919/blos/'
folder_vlos = '/home/ulyanov/data/pmi/20250919/vlos/'

In [7]:
def fwhm(sigma, gamma):
    fg = 2 * sigma * np.sqrt(2 * np.log(2))
    fl = 2 * gamma
    return 0.5343 * fl + np.sqrt(0.2169 * fl ** 2 + fg ** 2)

In [9]:
q_V = 299792458 / 6173.341

import fnmatch
from connect import bob

sftp = bob()

top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

folder = '/home/ulyanov/data/solo/phi/2026/'

for directory in dirs:
    if fnmatch.fnmatch(directory, '2025-09-19'):
        for file in sorted(sftp.listdir(top_dir + directory))[:]:
            if fnmatch.fnmatch(file, '*fdt-alam*C_*.fits.gz'):
                print(file)

                remote_file = top_dir + directory + '/' + file
                local_file = 'temp.fits.gz'
                sftp.get(remote_file, local_file)

                data, header = process(local_file,
                                       dark_file=dark_file,
                                       deadpix_file=deadpix_file,
                                       prefilter_file=prefilter_file,
                                       #cavity_file=cavity_file,
                                       flatfield_file=flat_file,
                                       ghost_file=ghost_file,
                                       distortion_file=distortion_file,
                                       _realign=True,
                                       _find_center=True,
                                       _demodulate=True,
                                       _correct_fringes=True,
                                       _correct_crosstalk=True,
                                       _calc_wavelengths=True,
                                       _mask=True,
                                       )

                #stop

                #data = rebin(data, 2, update_header=header)

                Blos, Vlos = classical_estimates(data, header, lam=0.1)

                file_blos = generate_filename(file, prefix='blos', folder=folder_blos)
                file_vlos = generate_filename(file, prefix='vlos', folder=folder_vlos)

                clone_fits(local_file, file_blos, Blos, header)
                clone_fits(local_file, file_vlos, Vlos, header)


solo_L1_phi-fdt-alam_20250919T051503_V202604242055C_0549190501.fits.gz


/home/ulyanov/PycharmProjects/flatfield/src/fit_voigt.py:43: RuntimeWarning: invalid value encountered in divide
  z = (xc + 1j * gamma) / sigma / np.sqrt(2)
/home/ulyanov/miniconda3/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


solo_L1_phi-fdt-alam_20250919T111503_V202604242055C_0549190502.fits.gz
solo_L1_phi-fdt-alam_20250919T171503_V202604242055C_0549190503.fits.gz
solo_L1_phi-fdt-alam_20250919T220003_V202604242053C_0549190515.fits.gz


In [37]:
plt.figure(figsize=(10,10))
plt.imshow(data[-1,3], 'gray', vmin=-30, vmax=30)
plt.tight_layout()

In [9]:
plt.figure(figsize=(10,10))
plt.imshow(Blos, 'seismic', vmin=-100, vmax=100)
plt.tight_layout()

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(params[0], 'seismic', vmin=-0.1, vmax=0.1)
plt.tight_layout()

In [15]:
plt.figure(figsize=(10,10))
plt.imshow(params[1], 'inferno', vmin=0.030, vmax=0.055)
plt.tight_layout()

In [11]:
plt.figure(figsize=(10,10))
plt.imshow(res, 'inferno', vmin=0, vmax=50)
plt.tight_layout()


In [12]:
plt.figure(figsize=(10,10))
plt.imshow(-params[3] / params[2], 'inferno', vmin=0.05, vmax=0.08)
plt.tight_layout()

In [11]:
plt.figure(figsize=(10,10))
plt.imshow(params[2], 'inferno')
plt.tight_layout()